# makegif

In [1]:
experiment_folder = "/home/dlee/code/Plasma/Argon_steady_New_data_2510/Steady_BC_Hard_12_colloc0_make_gif_gradnorm_updated"

In [2]:
# =========================
# [NEW] Snapshot / Plot config
# =========================
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")   # 노트북에서 화면 출력 방지 (파일로만 저장)
import matplotlib.pyplot as plt
import time

# (요청) 저장 루트: 1/2/3 기능 산출물 모두 이 폴더로
CKPT_SAVED_DIR = Path(experiment_folder+"/CKPT_SAVED")
CKPT_SAVED_DIR.mkdir(parents=True, exist_ok=True)

# (요청) 10만 epoch마다 별도 ckpt 저장
EXTRA_CKPT_EVERY = 1000000

# (요청) 5만 epoch마다 결과 npz 저장 (참고 설명에 맞춰 ckpt는 저장하지 않음)
NPZ_SAVE_EVERY = 1000

# (요청) 내부 플롯을 저장만 할 epoch들
PLOT_EPOCHS = [10000, 50000, 100000, 150000, 200000, 250000,300000]

# 파일명 유틸
def _tag(step: int) -> str:
    return f"step{step:07d}"

def _now():
    return time.strftime("%Y%m%d-%H%M%S")

# 0) 공통
def epoch_dir(step): return CKPT_SAVED_DIR / f"step_{step:07d}"

# parameter

In [3]:
# setting
# 자유롭게 바꿀 수 있는 값들
WIDTH   = 256
DEPTH   = 2 ## Pirate Net 블럭 개수 - pirate net은 x3 해서 계산되니까...
ACT     = 'swish'         # 'tanh', 'relu', 'gelu', 'swish', 'elu', 'sigmoid'…
EPOCHS  = 300000+1
N_r, N_b= 1000, 0
N_bpde = 0
N_lab   = 0          # csv 관측점 중 학습에 쓸 개수
LR      = 1.e-3
RESAMPLE_COLL_EVERY = 100   # ← (NEW) collocation 재샘플링 주기(스텝)

USE_PEXT = True
USE_DYNAMIC_UB = True  # True면 uB = ±sqrt(Te/mi_bar), False면 상수 u_B 사용

# ===== Causal (strict; eq.14–15) 설정 =====
CAUSAL_STRICT   = True         # 논문 causal training ON/OFF
CAUSAL_EPS      = 1.0         # ε in w(t) = exp(-ε * ∫_0^t ∫ |R|^2 dx dτ)
CAUSAL_CLIP_MIN = 1e-6         # 수치 안정성용

In [4]:
# ---- MUST RUN FIRST: set paths for JAX in this notebook ----
import os, sysconfig, ctypes, glob, pathlib

# 당신 venv의 site-packages 경로
SP = sysconfig.get_paths()["purelib"]

# JAX가 필요로 하는 라이브러리 경로를 맨 앞에 둠 (cusparse + nvjitlink + CUDA 12.1)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"
os.environ["LD_LIBRARY_PATH"]      = (
    f"{SP}/nvidia/cusparse/lib:"
    f"{SP}/nvidia/nvjitlink/lib:"
    "/usr/local/cuda-12.1/targets/x86_64-linux/lib:"
    "/usr/local/cuda/lib64"
)

# (권장) 선로드: nvJitLink -> cuSPARSE (RTLD_GLOBAL로 전체에 공개)
def _first(patts, base):
    for pat in patts:
        for p in sorted(pathlib.Path(base).glob(pat)):
            return str(p)
    return None

def _gl(path):
    if path: ctypes.CDLL(path, mode=getattr(ctypes, "RTLD_GLOBAL", 0))

_gl(_first(["libnvJitLink.so*", "libnvjitlink.so*"], f"{SP}/nvidia/nvjitlink/lib"))
_gl(_first(["libcusparse.so*", "libcusparse*.so*"],   f"{SP}/nvidia/cusparse/lib"))

print("[env ready] SP =", SP)

[env ready] SP = /home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages


In [5]:
import jax, jax.numpy as jnp
from jax.extend.backend import get_backend
print("[check] platform:", get_backend().platform)
print("[check] devices :", jax.devices())

ERROR:2025-11-25 15:31:52,019:jax._src.xla_bridge:487: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages/jax/_src/xla_bridge.py", line 485, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


[check] platform: cpu
[check] devices : [CpuDevice(id=0)]


In [6]:
import os
print(os.getcwd())
os.chdir(experiment_folder)
print(os.getcwd())

/home/dlee/code/Plasma/Argon_steady_New_data_2510/Steady_BC_Hard_12_colloc0_make_gif_gradnorm_updated
/home/dlee/code/Plasma/Argon_steady_New_data_2510/Steady_BC_Hard_12_colloc0_make_gif_gradnorm_updated


In [7]:
import os, sys
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
print("python =", sys.executable)

LD_LIBRARY_PATH = /home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages/nvidia/cusparse/lib:/home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages/nvidia/nvjitlink/lib:/usr/local/cuda-12.1/targets/x86_64-linux/lib:/usr/local/cuda/lib64
python = /home/dlee/code/Plasma/.venv_jax_local/bin/python


In [8]:
import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax import grad, vmap, jit, lax
import flax.linen as nn
import optax
from flax.training import train_state
import pandas as pd
import matplotlib.pyplot as plt
from flax.core import unfreeze, freeze
import optax
from soap_jax import soap
from flax import serialization


# from tqdm.notebook import trange   # 예쁜 progress bar
from tqdm.auto import trange      # ← 가장 안전

jax.config.update("jax_enable_x64", True)   # (선택) 배정밀도 사용
key = jax.random.PRNGKey(0)
DTYPE = jnp.float64


# ── (NEW) humanize NaN-safe monkey patch: Orbax 로그 포맷에서 NaN로 죽는 문제 방지
import math
import humanize.filesize as _fs
if not hasattr(_fs, "_orig_naturalsize"):
    _fs._orig_naturalsize = _fs.naturalsize
def _safe_naturalsize(value, *args, **kwargs):
    try:
        if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
            return "NaN"
        return _fs._orig_naturalsize(value, *args, **kwargs)
    except Exception:
        return "NaN"
_fs.naturalsize = _safe_naturalsize

/home/dlee/code/Plasma/.venv_jax_local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from pathlib import Path

# 위 셀: experiment_folder = "/home/dlee/code/Plasma/Argon_steady/Steady_BC_Hard_03_density_only"

exp = Path(experiment_folder).resolve()

# (고정) 데이터 절대경로
data_path = Path("/home/dlee/code/Plasma/Data/output_4_v_new.csv")

# (실험별) 체크포인트 경로
CKPT_DIR_NAME = "ckpts_mlp_soap_bcdata_saw_te_cos"   # 필요시 바꿔 쓰세요
CKPT_DIR  = exp / CKPT_DIR_NAME
BEST_PATH = CKPT_DIR / "best.npz"

CKPT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID" 
import re
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax import grad, vmap, jit, lax
import flax.linen as nn
import optax
from flax.training import train_state
import pandas as pd
import matplotlib.pyplot as plt
from flax.core import unfreeze, freeze
import optax
from soap_jax import soap
from flax import serialization


# from tqdm.notebook import trange   # 예쁜 progress bar
from tqdm.auto import trange      # ← 가장 안전

jax.config.update("jax_enable_x64", True)   # (선택) 배정밀도 사용
key = jax.random.PRNGKey(0)
DTYPE = jnp.float64


# ── (NEW) humanize NaN-safe monkey patch: Orbax 로그 포맷에서 NaN로 죽는 문제 방지
import math
import humanize.filesize as _fs
if not hasattr(_fs, "_orig_naturalsize"):
    _fs._orig_naturalsize = _fs.naturalsize
def _safe_naturalsize(value, *args, **kwargs):
    try:
        if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
            return "NaN"
        return _fs._orig_naturalsize(value, *args, **kwargs)
    except Exception:
        return "NaN"
_fs.naturalsize = _safe_naturalsize

In [11]:
from jax.extend.backend import get_backend
def _dev_of(x):
    # jax>=0.4에서 sharding이 있을 수 있어 안전하게 처리
    if hasattr(x, "device"):
        try:
            return x.device()
        except Exception:
            pass
    if hasattr(x, "devices"):
        try:
            return x.devices()
        except Exception:
            pass
    return "unknown"

print("[JAX] default_backend  :", jax.default_backend())
print("[JAX] backend.platform :", get_backend().platform)
print("[JAX] all devices      :", jax.devices())
print("[JAX] gpu devices      :", jax.devices("gpu"))

# 간단 연산을 JIT로 한 번 돌려서 실제 실행 디바이스 확인
@jax.jit
def _probe(y):
    return y * 2 + 1

_probe_in  = jnp.ones((8,), dtype=DTYPE)
_probe_out = _probe(_probe_in)

print("[JAX] array device     :", _dev_of(_probe_in))
print("[JAX] jit result device:", _dev_of(_probe_out))

[JAX] default_backend  : cpu
[JAX] backend.platform : cpu
[JAX] all devices      : [CpuDevice(id=0)]


RuntimeError: Unknown backend: 'gpu' requested, but no platforms that are instances of gpu are present. Platforms present are: cpu

## plot

In [12]:
def build_log_df(log):
    base_cols = [
        'step', 'total', 'phys_total', 'phys_r1', 'phys_r2', 'phys_r3', 'phys_r4',
        'bc_total', 'bc_flux_L', 'bc_flux_R', 'bc_dTe_L', 'bc_dTe_R',
        'bc_pde_flux_L', 'bc_pde_flux_R', 'bc_pde_dTe_L', 'bc_pde_dTe_R',
        'data_total', 'data_n', 'data_Te', 'data_E', 'data_G',
        'lambda_r1', 'lambda_r2', 'lambda_r3', 'lambda_r4', 'lambda_data',
    ]
    # sigma_cols = ['sigma_phys','sigma_bc_n','sigma_bc_T','sigma_data','sigma_ic']

    arr = []
    for row in log:
        arr.append([float(np.asarray(row[c])) for c in base_cols])

    # 길이에 따라 컬럼 구성
    ncol = len(arr[0]) if arr else 0
    # if ncol == len(base_cols) + len(sigma_cols):
        # cols = base_cols + sigma_cols
    if ncol == len(base_cols):
        cols = base_cols
    else:
        # 예상 밖이면 자동 생성
        cols = [f'col{i}' for i in range(ncol)]

    df = pd.DataFrame(arr, columns=cols)
    return df

In [13]:
# =========================
# [NEW] Snapshot helpers: npz 저장 + 플롯 저장 (표준화)
# =========================

def _df_to_arrays(df: pd.DataFrame):
    """DataFrame -> {col: np.ndarray} 변환 (npz에 직렬화하기 쉽게)"""
    out = {}
    for c in df.columns:
        out[f"log__{c}"] = np.asarray(df[c].to_numpy())
    out["log__columns"] = np.array(df.columns.to_list())
    return out

def _eval_overlay_arrays(state, data_path, tail_k=None):
    """
    현재 state로 CSV 지점에서 PINN 예측을 평가하고,
    Exact(CSV)과 PINN을 모두 반환 (npz 저장용).
    """
    df = pd.read_csv(data_path)
    # 기존 compare_profiles_steady와 동일한 방식으로 tail/final 스냅샷을 구성
    def _tail_average_by_z_np(df, tail_k=None):
        if (tail_k is None) or (tail_k <= 1):
            t_star = df['time[s]'].max()
            sub = df.loc[df['time[s]'] == t_star].copy()
            return t_star, sub.sort_values('z[m]')
        ut = np.sort(df['time[s]'].unique())
        sel = ut[-int(tail_k):]
        sub = df[df['time[s]'].isin(sel)].copy()
        g = sub.groupby('z[m]', as_index=False)[['ni_norm','Te_norm','E_norm','Fi_norm']].mean()
        return float(sel.mean()), g.sort_values('z[m]')

    t_star_SI, sub = _tail_average_by_z_np(df, tail_k=tail_k)

    z_SI   = sub['z[m]'].to_numpy()
    ni_sim = sub['ni_norm'].to_numpy()
    Te_sim = sub['Te_norm'].to_numpy()
    E_sim  = sub['E_norm'].to_numpy()
    G_sim  = sub['Fi_norm'].to_numpy()

    # PINN 예측
    z_bar = jnp.asarray(z_SI / l0, dtype=DTYPE)
    Np, Tep, Ep, Gp = jax.vmap(lambda zz: model_apply(state.params['nn'], zz))(z_bar)
    ni_pinn = np.asarray(jnp.exp(Np))
    Te_pinn = np.asarray(Tep)
    E_pinn  = np.asarray(Ep)
    G_pinn  = np.asarray(Gp)

    return {
        "z_SI": z_SI, "ni_sim": ni_sim, "Te_sim": Te_sim, "E_sim": E_sim, "G_sim": G_sim,
        "ni_pinn": ni_pinn, "Te_pinn": Te_pinn, "E_pinn": E_pinn, "G_pinn": G_pinn,
        "t_star_SI": np.array(t_star_SI, dtype=float),
    }

def _eval_grid_prediction(state, ngrid=512):
    """보조: 균일 z-grid에서 PINN 예측(시각화 편의용)."""
    z_bar_grid = np.linspace(0.0, float(Lz), ngrid, dtype=float)
    Np, Tep, Ep, Gp = jax.vmap(lambda zz: model_apply(state.params['nn'], zz))(jnp.asarray(z_bar_grid, dtype=DTYPE))
    return {
        "z_bar_grid": z_bar_grid,
        "n_hat_grid": np.exp(np.asarray(Np)),
        "Te_hat_grid": np.asarray(Tep),
        "E_hat_grid": np.asarray(Ep),
        "G_hat_grid": np.asarray(Gp),
    }

def _batch_to_arrays(batch):
    """현재 배치의 좌표들도 기록(필요 시 샘플링 산점도 재현 용)."""
    (z_r,
     z_b0, z_b1,
     z_lab, n_lab, Te_lab, E_lab, G_lab,
     n_b0_lab, Te_b0_lab, E_b0_lab, G_b0_lab,
     n_b1_lab, Te_b1_lab, E_b1_lab, G_b1_lab,
     z_bpde0, z_bpde1) = batch
    return {
        "z_r": np.asarray(z_r),
        "z_lab": np.asarray(z_lab),
        "z_b0": np.asarray(z_b0),
        "z_b1": np.asarray(z_b1),
        "z_bpde0": np.asarray(z_bpde0),
        "z_bpde1": np.asarray(z_bpde1),
        # 라벨도 혹시 플롯 재현 필요 시 기록
        "n_lab": np.asarray(n_lab),
        "Te_lab": np.asarray(Te_lab),
        "E_lab": np.asarray(E_lab),
        "G_lab": np.asarray(G_lab),
    }

def save_results_npz(state, log, batch, step: int, out_dir: Path, data_path: str, tail_k=None):
    """
    npz 저장: 
      - log 전체
      - PINN vs C-sim overlay 데이터
      - uniform grid 예측
      - batch (collocation/data) 좌표
      - meta 정보(step, hash, constants)
      - normalized z for exact data (z_bar_sim)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------------
    # Meta + constants
    # -------------------------------
    df_log = build_log_df(log)
    pack = {
        "meta__step": np.array(step, dtype=np.int64),
        "meta__saved_at": np.array(_now()),
        "meta__hash": np.array(hash_params(state.params)),
        "const__l0": np.array(l0),
        "const__t0": np.array(t0),
        "const__Lz": np.array(float(Lz)),
    }

    # -------------------------------
    # 1) 로그 전체 저장
    # -------------------------------
    pack.update(_df_to_arrays(df_log))

    # -------------------------------
    # 2) Exact vs PINN overlay arrays
    # -------------------------------
    overlay = _eval_overlay_arrays(state, data_path, tail_k=tail_k)
    pack.update(overlay)

    # z → normalized (편집용 ipynb에서 바로 사용 가능)
    pack["z_bar_sim"] = overlay["z_SI"] / l0

    # -------------------------------
    # 3) Uniform grid prediction
    # -------------------------------
    pack.update(_eval_grid_prediction(state, ngrid=512))

    # -------------------------------
    # 4) Batch (collocation / lab / boundary)
    # -------------------------------
    if batch is not None:
        pack.update(_batch_to_arrays(batch))

    # -------------------------------
    # 5) Save npz
    # -------------------------------
    npz_path = out_dir / f"results_{_tag(step)}.npz"
    np.savez(npz_path, **pack)
    print(f"[npz] saved → {npz_path}")

    return npz_path

# =========================
# [NEW] 학습 내부 저장용 플롯 (화면 출력 없이 파일로만)
# =========================
def _save_loss_plots(df_log: pd.DataFrame, out_dir: Path, step: int):
    out_dir.mkdir(parents=True, exist_ok=True)

    def _plot_and_save(ycols, title, ylog, fname):
        plt.figure(figsize=(7,4))
        for c in ycols:
            if c in df_log.columns:
                plt.plot(df_log['step'], df_log[c], label=c)
        plt.xlabel('step'); plt.ylabel('value'); 
        plt.title(title)
        if ylog: plt.yscale('log')
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
        plt.savefig(out_dir / fname, dpi=160)
        plt.close()

    # Plot 1: total/grouped
    cols = ['total', 'phys_total', 'bc_total', 'data_total']
    if 'ic_total' in df_log.columns and (np.nan_to_num(df_log['ic_total'].to_numpy()).sum() != 0.0):
        cols.append('ic_total')
    _plot_and_save(cols, '[Plot 1] Total & Grouped Sums (steady)', True, f"losses_plot1_{_tag(step)}.png")

    # Plot 2: physics breakdown
    _plot_and_save(['phys_total','phys_r1','phys_r2','phys_r3','phys_r4'],
                   '[Plot 2] Physics Residual Breakdown (steady)', True, f"losses_plot2_{_tag(step)}.png")

    # Plot 3: physics weights
    _plot_and_save(['phys_w1','phys_w2','phys_w3','phys_w4'],
                   '[Plot 3] Physics Residual weight (steady)', False, f"losses_plot3_{_tag(step)}.png")

    # Plot 4: SAW lambdas
    lam_cols = ['lambda_phys', 'lambda_bc', 'lambda_data']
    if 'lambda_ic' in df_log.columns and (np.nan_to_num(df_log['lambda_ic'].to_numpy()).sum() != 0.0):
        lam_cols.append('lambda_ic')
    _plot_and_save(lam_cols, '[Plot 4] SAW λ (steady)', True, f"losses_plot4_{_tag(step)}.png")

    # Plot 5: data breakdown
    _plot_and_save(['data_total','data_n','data_Te','data_E','data_G'],
                   '[Plot 5] Data Loss Breakdown (steady)', True, f"losses_plot5_{_tag(step)}.png")

def _save_overlay_plots_from_arrays(overlay: dict, out_dir: Path, step: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    z_SI   = overlay["z_SI"]
    ni_sim = overlay["ni_sim"]; Te_sim = overlay["Te_sim"]
    E_sim  = overlay["E_sim"];  G_sim  = overlay["G_sim"]
    ni_p   = overlay["ni_pinn"]; Te_p  = overlay["Te_pinn"]
    E_p    = overlay["E_pinn"];  G_p   = overlay["G_pinn"]
    tag = f"(t≈{overlay['t_star_SI']:.2e} s)"

    def _ov(x, y1, y2, ylab, title, fname):
        plt.figure(figsize=(7,4))
        plt.plot(x, y1, marker='o', ms=3, lw=0, label='Exact(C-sim)')
        plt.plot(x, y2, lw=1.8, label='PINN')
        plt.xlabel('z [m]'); plt.ylabel(ylab); plt.title(f"{title} {tag}")
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
        plt.savefig(out_dir / fname, dpi=160); plt.close()

    _ov(z_SI, ni_sim, ni_p,  r"$n_i/n_0$", "Ion Density", f"overlay_ni_{_tag(step)}.png")
    _ov(z_SI, Te_sim, Te_p,  r"$T_e/T_0$", "Electron Temperature", f"overlay_Te_{_tag(step)}.png")
    _ov(z_SI, E_sim,  E_p,   r"$E$",       "Electric Field", f"overlay_E_{_tag(step)}.png")
    _ov(z_SI, G_sim,  G_p,   r"$\Gamma$",  "Ion Flux", f"overlay_G_{_tag(step)}.png")

def save_all_plots_silent(state, log, step: int, out_root: Path, data_path: str):
    """(요청 3) 학습 도중 지정 epoch마다 플롯 생성하되 화면에 출력하지 않고 저장만."""
    plots_dir = out_root / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    df_log = build_log_df(log)
    _save_loss_plots(df_log, plots_dir, step)
    overlay = _eval_overlay_arrays(state, data_path, tail_k=None)
    _save_overlay_plots_from_arrays(overlay, plots_dir, step)
    print(f"[plot] saved loss/overlay plots at {_tag(step)} → {plots_dir}")

In [14]:
#──────────────────────────────────────────────────────────
# 1) CONSTANTS  (SI → 무차원)  — PlasmaPDE에 전달할 config.phys 구성
# ──────────────────────────────────────────────────────────

eps = 1.e-12
# ① SI 입력값
PRS_mTorr = 30.0          # [mTorr]
Prf_W     = 500.0         # [W]
Lp_m      = 0.10          # [m]
Rp_m      = 0.25          # [m]
Ti_SI     = 300.0         # [K]
Tg_SI     = 300.0         # [K]
nN_SI     = 3.22e19 * PRS_mTorr  # [m^-3]

# ② 정규화 기준
n0  = 15.0e16              # [m^-3]
T0  = 11604.0             # [K]  (=1 eV / kB)
t0  = 1.0e-8              # [s]
m0  = 1.6605e-27          # [kg] (=1 amu)
kB  = 1.3807e-23          # [J/K]
QE  = 1.6022e-19          # [C]

u0 = jnp.sqrt(kB*T0/m0)   # [m/s]
l0 = u0 * t0              # [m]
k0 = 1.0 / (t0 * n0)      # [m^3 s^-1]
E0 = kB*T0 / (QE*l0)      # [V/m]
R0 = k0 * n0**2           # [m^-3 s^-1]
P0 = n0*kB*T0*(l0**3)/t0  # [W]

# ③ 무차원 값
Lz         = Lp_bar  = Lp_m / l0
Rp_bar  = Rp_m / l0
m_e_bar    = me_bar  = 9.1094e-31 / m0
m_i_bar    = m_beta_bar = mi_bar  = 39.948
T_i_bar    = Ti_bar  = Ti_SI / T0
T_beta_bar = Tg_bar  = Tg_SI / T0
nN_bar  = nN_SI / n0
Prf_bar = Prf_W / P0
dt_bar  = 1.0            # (dt == t0 라서)


# ④ 반응항 상수 (Ṙ = C_R · n̂ · Tê^0.6329 · exp(−α/Tê))
const_Reac = 5.779e-17   # [m^3 s^-1 K^-0.6329]
Eion_K     = 1.864e5     # [K]
C_R        = nN_bar * const_Reac * (T0**0.6329) / k0
alpha      = Eion_K / T0

In [15]:
# ──────────────────────────────────────────────────────────
# 2) PlasmaPDE 구성에 넣을 phys 딕셔너리
# ──────────────────────────────────────────────────────────
phys = dict(
    # 정규화 기준
    n0=float(n0), T0=float(T0), t0=float(t0), m0=float(m0), kB=float(kB), QE=float(QE),

    # 무차원 파라미터
    mi_bar=float(mi_bar), Ti_bar=float(Ti_bar), Tbeta_bar=float(Tg_bar),
    me_bar=float(me_bar), m_beta_bar=float(mi_bar),

    # 기하/전력
    Lp_bar=float(Lp_bar), Rp_bar=float(Rp_bar), Prf_bar=float(Prf_bar),
    PRS_mTorr=float(PRS_mTorr), nN_bar=float(nN_bar),

    # 반응항
    const_Reac=float(const_Reac), Eion_K=float(Eion_K), C_R=float(C_R), alpha=float(alpha),
)


In [16]:
# =========================
# Steady-state data loader
# =========================
def load_csv_steady(path, t_mode: str = "final", tail_k: int | None = None,
                    t_value: float | None = None, tol: float = 1e-12):
    """
    path: CSV 경로
    t_mode: "final"(기본) 또는 "fixed" (t_value가 주어질 때)
    tail_k: 마지막 K개 시점의 평균을 쓸 때 지정 (None/1 이면 단일 시점)
    t_value: t_mode="fixed"일 때 사용할 시점
    tol: 시점 선택 isclose 허용오차 (무차원 time 기준)
    필요한 상수: t0, l0, DTYPE (기존 코드와 동일)
    """
    df = pd.read_csv(path)

    t = jnp.array(df["time[s]"], dtype=DTYPE) / t0
    z = jnp.array(df["z[m]"],    dtype=DTYPE) / l0
    n = jnp.array(df["ni_norm"], dtype=DTYPE)
    Te = jnp.array(df["Te_norm"], dtype=DTYPE)
    E  = jnp.array(df["E_norm"],  dtype=DTYPE)
    G  = jnp.array(df["Fi_norm"], dtype=DTYPE)

    if tail_k is None or tail_k == 1:
        # 단일 시점 스냅샷
        if t_mode == "final" or (t_mode == "fixed" and t_value is None):
            t_star = jnp.max(t).item()
        elif t_mode == "fixed":
            t_star = float(t_value)
        else:
            raise ValueError("t_mode must be 'final' or 'fixed'.")

        mask = jnp.isclose(t, t_star, atol=tol)
        z_ss, n_ss, Te_ss, E_ss, G_ss = z[mask], n[mask], Te[mask], E[mask], G[mask]

    else:
        # 마지막 tail_k 시점 평균 (pandas로 안정적으로 groupby)
        _df = pd.DataFrame({
            "t":  jnp.array(t),
            "z":  jnp.array(z),
            "n":  jnp.array(n),
            "Te": jnp.array(Te),
            "E":  jnp.array(E),
            "G":  jnp.array(G),
        })
        uniq_t = jnp.sort(_df["t"].unique())
        sel_t  = uniq_t[-int(tail_k):]
        df_tail = _df[_df["t"].isin(sel_t)]
        df_ss = df_tail.groupby("z", as_index=False).mean(numeric_only=True)
        z_ss  = jnp.array(df_ss["z"],  dtype=DTYPE)
        n_ss  = jnp.array(df_ss["n"],  dtype=DTYPE)
        Te_ss = jnp.array(df_ss["Te"], dtype=DTYPE)
        E_ss  = jnp.array(df_ss["E"],  dtype=DTYPE)
        G_ss  = jnp.array(df_ss["G"],  dtype=DTYPE)
        t_star = float(sel_t.mean())

    # z 기준 정렬
    idx = jnp.argsort(z_ss)
    z_ss, n_ss, Te_ss, E_ss, G_ss = z_ss[idx], n_ss[idx], Te_ss[idx], E_ss[idx], G_ss[idx]

    z_minmax = jnp.array([z_ss.min().item(), z_ss.max().item()])
    zfinal   = z_minmax[1].item()
    N_data_ss = int(z_ss.shape[0])

    print(f"[steady] t*={t_star:.6g}, {N_data_ss} labelled points")
    # 하위 코드들과의 호환을 위해 t 관련 상수도 채워둠(실제론 쓰지 않음)
    t_minmax = jnp.array([t_star, t_star])
    Tfinal   = t_star

    return (z_ss, n_ss, Te_ss, E_ss, G_ss), z_minmax, zfinal, t_minmax, Tfinal



(z_ss, n_ss, Te_ss, E_ss, G_ss), z_minmax, zfinal, t_minmax, Tfinal = load_csv_steady(
    data_path, t_mode="final", tail_k=None  # tail_k=5 같은 값으로 마지막 5개 평균도 가능
)
N_data = z_ss.shape[0]
print("loaded steady:", N_data)


# ========== 여기에 추가 ==========
# Hard BC를 위한 경계값 추출
def get_boundary_values(z_ss, n_ss, Te_ss, E_ss, G_ss):
    """경계값 추출 (z=0과 z=L에서)"""
    idx_0 = jnp.argmin(jnp.abs(z_ss))  # z=0에 가장 가까운 점
    idx_L = jnp.argmax(z_ss)           # z=L 점
    
    return {
        'n_0': n_ss[idx_0], 'n_L': n_ss[idx_L],
        'Te_0': Te_ss[idx_0], 'Te_L': Te_ss[idx_L],
        'E_0': E_ss[idx_0], 'E_L': E_ss[idx_L],
        'G_0': G_ss[idx_0], 'G_L': G_ss[idx_L],
    }

BC_VALUES = get_boundary_values(z_ss, n_ss, Te_ss, E_ss, G_ss)
print(f"Boundary values: n(0)={BC_VALUES['n_0']:.3e}, n(L)={BC_VALUES['n_L']:.3e}")
# ==================================

[steady] t*=50000, 51 labelled points
loaded steady: 51
Boundary values: n(0)=8.415e-01, n(L)=7.841e-01


In [17]:
# =========================
# 1D helpers (z 전용)
# =========================
def _uniform(key, n, low, high):
    u = jax.random.uniform(key, (n,), dtype=DTYPE)
    return low + (high - low) * u

def _left_heavy(key, n, low, high, power: float = 2.0):
    u = jax.random.uniform(key, (n,), dtype=DTYPE)
    return low + (high - low) * (u ** power)

def _cheb_symmetric(key, n, low, high):
    u = jax.random.uniform(key, (n,), dtype=DTYPE)
    x01 = 0.5 * (1.0 - jnp.cos(jnp.pi * u))
    return low + (high - low) * x01

def _sample_1d(key, n, low, high, mode: str = "uniform", power: float = 2.0):
    if mode == "uniform":
        return _uniform(key, n, low, high)
    elif mode == "left_heavy":
        return _left_heavy(key, n, low, high, power=power)
    elif mode == "cheb":
        return _cheb_symmetric(key, n, low, high)
    else:
        raise ValueError(f"Unknown mode: {mode}")

# =========================
# Steady-state sampler
# =========================
def sample_training_points_steady(
    key, N_r, N_b, N_lab: int = 0,
    *,
    z_minmax, z_data, n_data, Te_data, E_data, G_data,
    use_chebyshev: bool = False,
    random_mode: str = "uniform",
    z_power: float = 2.0,
    use_boundary_data: bool = False,
    boundary_tol: float = 1e-12,
    N_bpde: int = 0,
    z_r_fixed: jnp.ndarray | None = None, # (NEW) collocation 고정 입력 (주어지면 새 샘플링을 생략)
):
    """
    반환:
      (z_r,
       z_b0, z_b1,
       z_lab, n_lab, Te_lab, E_lab, G_lab,
       n_b0_lab, Te_b0_lab, E_b0_lab, G_b0_lab,
       n_b1_lab, Te_b1_lab, E_b1_lab, G_b1_lab,
       z_bpde0, z_bpde1)
    """
    mode_z_r = "cheb" if use_chebyshev else random_mode

    # RNG
    k1, k2, k3, k4 = jax.random.split(key, 4)

    z0, z1 = float(z_minmax[0]), float(z_minmax[1])

    # # 1) collocation in z
    # z_r = _sample_1d(k1, N_r, z0, z1, mode=mode_z_r, power=z_power)
    # 1) collocation in z — 고정 포인트가 있으면 그대로 사용
    if z_r_fixed is not None:
        assert z_r_fixed.ndim == 1 and z_r_fixed.shape[0] == N_r, \
            f"z_r_fixed must be 1D and length N_r={N_r}"
        z_r = z_r_fixed.astype(DTYPE)
    else:
        z_r = _sample_1d(k1, N_r, z0, z1, mode=mode_z_r, power=z_power)

    # 2) boundary points
    N_b0 = N_b // 2
    N_b1 = N_b - N_b0

    if use_boundary_data:
        b0_mask = jnp.isclose(z_data, 0.0, atol=boundary_tol)
        b1_mask = jnp.isclose(z_data, z1,  atol=boundary_tol)

        z_b0_all = z_data[b0_mask]; z_b1_all = z_data[b1_mask]
        n_b0_all, Te_b0_all, E_b0_all, G_b0_all = n_data[b0_mask], Te_data[b0_mask], E_data[b0_mask], G_data[b0_mask]
        n_b1_all, Te_b1_all, E_b1_all, G_b1_all = n_data[b1_mask], Te_data[b1_mask], E_data[b1_mask], G_data[b1_mask]

        if z_b0_all.shape[0] == 0:
            raise ValueError("No boundary labels at z=0; relax boundary_tol or disable use_boundary_data.")
        if z_b1_all.shape[0] == 0:
            raise ValueError("No boundary labels at z=L; relax boundary_tol or disable use_boundary_data.")

        idx_b0 = jax.random.choice(k2, z_b0_all.shape[0], shape=(N_b0,),
                                   replace=(N_b0 > z_b0_all.shape[0]))
        idx_b1 = jax.random.choice(k3, z_b1_all.shape[0], shape=(N_b1,),
                                   replace=(N_b1 > z_b1_all.shape[0]))

        z_b0 = z_b0_all[idx_b0]; z_b1 = z_b1_all[idx_b1]
        n_b0_lab, Te_b0_lab, E_b0_lab, G_b0_lab = n_b0_all[idx_b0], Te_b0_all[idx_b0], E_b0_all[idx_b0], G_b0_all[idx_b0]
        n_b1_lab, Te_b1_lab, E_b1_lab, G_b1_lab = n_b1_all[idx_b1], Te_b1_all[idx_b1], E_b1_all[idx_b1], G_b1_all[idx_b1]

    else:
        # 라벨 없이 BC 좌표만 제공 (PDE/약한 BC 용)
        z_b0 = jnp.zeros((N_b0,), dtype=DTYPE)
        z_b1 = jnp.full((N_b1,), z1, dtype=DTYPE)
        n_b0_lab = Te_b0_lab = E_b0_lab = G_b0_lab = jnp.zeros((0,), dtype=DTYPE)
        n_b1_lab = Te_b1_lab = E_b1_lab = G_b1_lab = jnp.zeros((0,), dtype=DTYPE)

    # 3) optional interior labels
    if N_lab and N_lab > 0:
        idx_lab = jax.random.choice(k4, z_data.shape[0], shape=(N_lab,), replace=False)
        z_lab  = z_data[idx_lab]
        n_lab  = n_data[idx_lab]
        Te_lab = Te_data[idx_lab]
        E_lab  = E_data[idx_lab]
        G_lab  = G_data[idx_lab]
    else:
        z_lab  = jnp.zeros((0,), dtype=DTYPE)
        n_lab  = jnp.zeros((0,), dtype=DTYPE)
        Te_lab = jnp.zeros((0,), dtype=DTYPE)
        E_lab  = jnp.zeros((0,), dtype=DTYPE)
        G_lab  = jnp.zeros((0,), dtype=DTYPE)

    # 4) PDE용 경계 collocation (잔차만 강제, 라벨 없음)
    if N_bpde and N_bpde > 0:
        N_p0 = N_bpde // 2
        N_p1 = N_bpde - N_p0
        z_bpde0 = jnp.zeros((N_p0,), dtype=DTYPE)
        z_bpde1 = jnp.full((N_p1,), z1, dtype=DTYPE)
    else:
        z_bpde0 = jnp.zeros((0,), dtype=DTYPE)
        z_bpde1 = jnp.zeros((0,), dtype=DTYPE)

    return (z_r,
            z_b0, z_b1,
            z_lab, n_lab, Te_lab, E_lab, G_lab,
            n_b0_lab, Te_b0_lab, E_b0_lab, G_b0_lab,
            n_b1_lab, Te_b1_lab, E_b1_lab, G_b1_lab,
            z_bpde0, z_bpde1)

key = jax.random.PRNGKey(0)

(z_r,
 z_b0, z_b1,
 z_lab, n_lab, Te_lab, E_lab, G_lab,
 n_b0_lab, Te_b0_lab, E_b0_lab, G_b0_lab,
 n_b1_lab, Te_b1_lab, E_b1_lab, G_b1_lab,
 z_bpde0, z_bpde1) = sample_training_points_steady(
    key, N_r=N_r, N_b=N_b, N_lab=N_lab,
    z_minmax=z_minmax,
    z_data=z_ss, n_data=n_ss, Te_data=Te_ss, E_data=E_ss, G_data=G_ss,
    use_chebyshev=True, random_mode="uniform", z_power=2.0,
    use_boundary_data=False, boundary_tol=1e-12,
    N_bpde=2
)


In [18]:

plt.figure(figsize = (7,2))
plt.scatter(z_r,  5.e-4*jnp.ones(z_r.shape), s=16, alpha = 0.5, marker = '.', label = 'collocation')
plt.scatter(z_lab,5.e-4*jnp.ones(z_lab.shape), s=64, alpha = 0.8, marker = 'x',c='r', label = 'labeled data')
plt.scatter(z_b0, 5.e-4*jnp.ones(z_b0.shape), s=16, alpha = 1.0,c='k', marker = '^', label = 'BC z=0')
plt.scatter(z_b1, 5.e-4*jnp.ones(z_b1.shape), s=16, alpha = 1.0,c='k', marker = 'v', label = 'BC z=zfinal')
plt.xlabel('z (normalized)')
plt.ylabel('t (normalized)')
plt.legend(loc='best', ncol=2)
plt.tight_layout()
plt.show()

/tmp/ipykernel_1997073/3280960916.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# ──────────────────────────────────────────────────────────
# 2) Pirate Network (Flax) — steady-mode 지원 (z-only)
# ──────────────────────────────────────────────────────────
import jax
import jax.numpy as jnp
import jax.nn as jnn
from jax import random
from flax import linen as nn
from flax.linen.initializers import glorot_normal, zeros
from typing import Callable, Dict, Union, Optional

DTYPE = jnp.float64

# -------------------------
# Random Weight Factorization
# -------------------------
def _weight_fact(init_fn: Callable, mean: float, stddev: float):
    """jaxpi 스타일 RWF: kernel 파라미터를 (g, v)로 등록."""
    def init(key, shape):
        key1, key2 = random.split(key)
        w = init_fn(key1, shape)                              # (in, out)
        g = jax.nn.initializers.normal(stddev)(key2, (shape[-1],)) + mean  # (out,)
        g = jnp.exp(g).astype(w.dtype)
        v = (w / g).astype(w.dtype)                           # (in, out) / (out,)
        return g, v
    return init

class Dense(nn.Module):
    features: int
    kernel_init: Callable = glorot_normal()
    bias_init: Callable = zeros
    reparam: Union[None, Dict] = None
    dtype: any = DTYPE
    param_dtype: any = DTYPE

    @nn.compact
    def __call__(self, x):
        in_dim = x.shape[-1]
        out_dim = self.features

        if self.reparam is None:
            kernel = self.param("kernel", self.kernel_init, (in_dim, out_dim)).astype(self.dtype)
        elif self.reparam["type"] == "weight_fact":
            g, v = self.param(
                "kernel",
                _weight_fact(self.kernel_init,
                             mean=self.reparam.get("mean", 1.0),
                             stddev=self.reparam.get("stddev", 0.1)),
                (in_dim, out_dim),
            )
            kernel = (g * v).astype(self.dtype)  # (out,) * (in,out)
        else:
            raise ValueError(f"Unknown reparam type: {self.reparam}")

        bias = self.param("bias", self.bias_init, (out_dim,)).astype(self.dtype)
        y = jnp.dot(x.astype(self.dtype), kernel) + bias
        return y

ACTS = {
    'tanh': jnn.tanh, 'relu': jnn.relu, 'gelu': jnn.gelu,
    'swish': jnn.swish, 'elu': jnn.elu, 'sigmoid': jnn.sigmoid, 'sin': jnp.sin
}

class PirateBlock(nn.Module):
    width: int
    act: str
    reparam: Union[None, Dict] = None
    
    @nn.compact
    def __call__(self, x, U, V):
        f = ACTS[self.act]
        h1 = f(Dense(self.width, dtype=DTYPE, param_dtype=DTYPE, reparam=self.reparam)(x))
        z1 = h1 * U + (1.0 - h1) * V
        h2 = f(Dense(self.width, dtype=DTYPE, param_dtype=DTYPE, reparam=self.reparam)(z1))
        z2 = h2 * U + (1.0 - h2) * V
        h  = f(Dense(self.width, dtype=DTYPE, param_dtype=DTYPE, reparam=self.reparam)(z2))
        alpha = self.param('alpha', nn.initializers.zeros, (), DTYPE)
        return alpha * h + (1.0 - alpha) * x
    
class PiratePINN_Network(nn.Module):
    width: int
    n_block: int
    act: str = 'tanh'
    # RFF
    fft: bool = False
    B: int = 64
    freq: float = 1.0
    # 좌표 정규화
    coord_norm: bool = False
    z_minmax: Optional[jnp.ndarray] = None
    # RWF 설정
    reparam: Union[None, Dict] = None

    def setup(self):
        in_dim = 1  # ★ steady 전용: z만 입력
        if self.fft:
            self.B_mat = self.param(
                'B_mat',
                jax.nn.initializers.normal(stddev=jnp.sqrt(2.0)),
                (self.B, in_dim), DTYPE
            )
        self.gate_U = Dense(self.width, dtype=DTYPE, param_dtype=DTYPE, reparam=self.reparam)
        self.gate_V = Dense(self.width, dtype=DTYPE, param_dtype=DTYPE, reparam=self.reparam)
        self.blocks = [PirateBlock(self.width, self.act, reparam=self.reparam)
                       for _ in range(self.n_block)]
        self.out = Dense(4, dtype=DTYPE, param_dtype=DTYPE, reparam=None)

    def _normalize(self, x):
        if not self.coord_norm:
            return x
        assert self.z_minmax is not None, "z_minmax must be set when coord_norm=True."
        z0, z1 = self.z_minmax
        z = x[..., 0]
        z_ = (z - z0) / (z1 - z0 + 1e-12)
        return z_[..., None]

    def _fourier_embed(self, x):
        proj = 2*jnp.pi*self.freq*(x @ self.B_mat.T)  # (N,B)
        return jnp.concatenate([jnp.sin(proj), jnp.cos(proj)], axis=-1)

    def __call__(self, x):
        # x: (..., 1)
        x = self._normalize(x) if self.coord_norm else x
        Φ = self._fourier_embed(x) if self.fft else x

        f = ACTS[self.act]
        U = f(self.gate_U(Φ))
        V = f(self.gate_V(Φ))

        h = Φ
        for blk in self.blocks:
            h = blk(h, U, V)

        y_raw = self.out(h)   # (..., 4) = [Ni_raw, Te_raw, E_raw, Gi_raw]
        return h, y_raw


In [20]:
from dataclasses import dataclass

@dataclass
class NetCfg:
    use_rwf: bool = False      # ← 여기만 False면 RWF 완전 OFF
    rwf_mean: float = 1.0
    rwf_std: float  = 0.1

def reparam_from_cfg(cfg: NetCfg):
    return {"type": "weight_fact", "mean": cfg.rwf_mean, "stddev": cfg.rwf_std} if cfg.use_rwf else None

cfg = NetCfg(use_rwf=True)  # ← RWF On

model = PiratePINN_Network(
    width=WIDTH, n_block=DEPTH, act=ACT,
    fft=True, B=WIDTH//2, freq=1.0,                 # RFF with N(0,2) inside
    coord_norm=True, z_minmax=z_minmax,
    reparam=reparam_from_cfg(cfg)                      # ← 여기!
)

def _vars_for_apply(params):
    # params가 {'nn': variables} 이면 variables만 꺼내고,
    # 이미 variables(dict with 'params' collection)면 그대로 사용
    return params['nn'] if (isinstance(params, dict) and 'nn' in params) else params

# def model_apply(params, z):
#     variables = _vars_for_apply(params)
#     inputs = jnp.asarray(z, dtype=DTYPE)[..., None]   # (N,1)
#     _, y_raw = model.apply(variables, inputs)
#     Ni_raw = y_raw[..., 0].astype(DTYPE)             # log n̂
#     Te_hat = jnp.exp(y_raw[..., 1].astype(DTYPE))    # Tê
#     E_raw  = y_raw[..., 2].astype(DTYPE)
#     Gi_raw = y_raw[..., 3].astype(DTYPE)
#     return Ni_raw, Te_hat, E_raw, Gi_raw
# 기존 model_apply 함수를 다음으로 교체:
def model_apply_hard_bc(params, z):
    """Hard boundary constraint가 적용된 model apply"""
    variables = _vars_for_apply(params)
    
    # 스칼라/벡터 입력 처리
    z = jnp.asarray(z, dtype=DTYPE)
    z_shape = z.shape
    z_flat = z.flatten()
    
    # 현재 위치에서의 네트워크 출력
    inputs = z_flat[..., None]
    _, y_raw = model.apply(variables, inputs)
    
    # 경계에서의 네트워크 출력
    _, y_0 = model.apply(variables, jnp.zeros((1,1), dtype=DTYPE))
    _, y_L = model.apply(variables, jnp.ones((1,1), dtype=DTYPE) * Lz)
    
    # 정규화된 좌표
    z_norm = z_flat / Lz
    
    # 경계 조건 (log scale for n and Te)
    bc_0 = jnp.array([jnp.log(BC_VALUES['n_0']), 
                      jnp.log(BC_VALUES['Te_0']),
                      BC_VALUES['E_0'],
                      BC_VALUES['G_0']], dtype=DTYPE)
    
    bc_L = jnp.array([jnp.log(BC_VALUES['n_L']),
                      jnp.log(BC_VALUES['Te_L']),
                      BC_VALUES['E_L'],
                      BC_VALUES['G_L']], dtype=DTYPE)
    
    # Hard constraint 적용
    correction_0 = (bc_0 - y_0[0])
    correction_L = (bc_L - y_L[0])
    
    # Broadcasting을 위한 reshape
    z_norm_exp = z_norm[:, None]
    y_corrected = y_raw + (1-z_norm_exp) * correction_0 + z_norm_exp * correction_L
    
    # 출력 변환
    Ni_raw = y_corrected[:, 0].reshape(z_shape)
    Te_hat = jnp.exp(y_corrected[:, 1]).reshape(z_shape)
    E_raw = y_corrected[:, 2].reshape(z_shape)
    Gi_raw = y_corrected[:, 3].reshape(z_shape)
    
    return Ni_raw, Te_hat, E_raw, Gi_raw

# 기존 model_apply를 대체
model_apply = model_apply_hard_bc

# ion-neutral collision (무차원 상수) — model.py와 동일식
def compute_nu_i_bar_const():
    mi_SI = m_i_bar * m0
    Ti_SI = T_i_bar * T0
    k0_SI = 1.0/(t0*n0)
    return nN_bar * (50e-20 * jnp.sqrt(8.0*kB*Ti_SI/(jnp.pi*mi_SI))) / k0_SI

nu_i_bar_const = compute_nu_i_bar_const()  # 상수

# electron-neutral collision (무차원, Te-의존)
def nu_eb_bar_fn(Te_hat):
    # Te_hat: 무차원(K/T0)
    Te_SI = Te_hat * T0
    k0_SI = 1.0/(t0*n0)
    return nN_bar * (3.5665e-15 * (Te_SI**0.3785) * jnp.exp(-2.292e4/(Te_SI + 1e-30))) / k0_SI

# 반응항 (무차원)  Ṙ = C_R · n̂ · Tê^0.6329 · exp(−α/Tê)
# C_R, alpha는 네가 constants에서 이미 정의했다고 가정
def R_tilde(n_hat, Te_hat):
    return C_R * n_hat * (Te_hat**0.6329) * jnp.exp(-alpha/(Te_hat + 1e-30))
Z_GRID = jnp.linspace(0.0, Lz, 10, dtype=DTYPE)  # Lz는 z 최대(무차원 길이)

def Pext_bar_fn(params, z, z_all=None):
    if z_all is None:
        z_all = Z_GRID
    def n_at(zz):
        N_hat, _, _, _ = model_apply(params, zz)
        return jnp.exp(N_hat)
    n_line = jax.vmap(n_at)(z_all)
    ave_ni = lax.stop_gradient(jnp.mean(n_line))

    ME, EPS0, CV = 9.1094e-31, 8.8542e-12, 2.9979e8
    omega_p = jnp.sqrt(ave_ni*n0 * (QE**2)/(ME*EPS0))
    delta_m = CV/(omega_p + 1e-30)

    u0_loc = jnp.sqrt(kB*T0/m0); l0_loc = u0_loc * t0
    delta_n = delta_m / (l0_loc + 1e-30)

    num = 2.0 * Prf_bar * jnp.exp(-2.0*z/(delta_n + 1e-30))
    den = (delta_n + 1e-30) * (1.0 - jnp.exp(-2.0*Lz/(delta_n + 1e-30))) * (jnp.pi * Rp_bar * Rp_bar)
    return (num / den).astype(DTYPE)

# def gamma_tilde(params, z):
#     N_hat, Te_hat, E_hat, G_hat = model_apply(params, z)
#     n_hat = jnp.exp(N_hat)
#     return G_hat, E_hat, N_hat, Te_hat, n_hat

# v_gamma_tilde = jax.vmap(gamma_tilde, in_axes=(None, 0))

# 기존 gamma_tilde 함수를 교체:
def gamma_tilde(params, z):
    N_hat, Te_hat, E_hat, G_hat = model_apply(params, z)  # 이미 hard BC 적용됨
    n_hat = jnp.exp(N_hat)
    return G_hat, E_hat, N_hat, Te_hat, n_hat

v_gamma_tilde = jax.vmap(gamma_tilde, in_axes=(None, 0))

def residuals(params, z):
    G_hat, E_hat, N_hat, Te_hat, n_hat = gamma_tilde(params, z)

    dG_dz = jax.grad(lambda zz: gamma_tilde(params, zz)[0])(z)
    R_hat = R_tilde(n_hat, Te_hat)
    r_1   = dG_dz - R_hat

    mu_i  = 1.0 / (m_i_bar * (nu_i_bar_const + eps))
    dN_dz = jax.grad(lambda zz: model_apply(params, zz)[0])(z)
    r_2   = G_hat - n_hat * mu_i * (E_hat - T_i_bar * dN_dz)

    dTe_dz = jax.grad(lambda zz: model_apply(params, zz)[1])(z)
    r_3    = E_hat + (Te_hat * dN_dz + dTe_dz)

    def GTe_space(zz):
        G_, E_, N_, Te_, n_ = gamma_tilde(params, zz)
        return 2.5 * G_ * Te_
    d_GTe_dz = jax.grad(GTe_space)(z)

    def kappa_at(Te_, n_):
        return 2.5 * (n_ * Te_) / (m_e_bar * (nu_eb_bar_fn(Te_) + eps))

    d_q_dz = jax.grad(
        lambda zz: kappa_at(model_apply(params, zz)[1],
                            jnp.exp(model_apply(params, zz)[0]))
                   * jax.grad(lambda zzz: model_apply(params, zzz)[1])(zz)
    )(z)

    Pext  = Pext_bar_fn(params, z)
    nu_eN = nu_eb_bar_fn(Te_hat)
    Ei    = 15.6 * R_hat

    energy_rhs = Pext - (E_hat * G_hat) \
                 - 3.0 * (m_e_bar / m_beta_bar) * n_hat * nu_eN * (Te_hat - T_beta_bar) \
                 - Ei

    r_4 = d_GTe_dz - d_q_dz - energy_rhs
    return r_1, r_2, r_3, r_4

v_residuals = jax.vmap(residuals, in_axes=(None, 0))

In [21]:
def _sort_by_time(t, *arrays):
    """t 오름차순 정렬. arrays도 동일하게 정렬."""
    idx = jnp.argsort(t.reshape(-1))
    return (t.reshape(-1)[idx],) + tuple(a.reshape(-1)[idx] for a in arrays)

def _causal_temporal_weights_strict(residual_sq_series, eps):
    """
    residual_sq_series: (N,), 이산화된 r_i = ∫Ω |R|^2 dx 값들(시간순!)
    w_i = exp( -ε * mean_{k<i} r_k ), w_1=1  (eq.15)
    """
    N      = residual_sq_series.shape[0]
    csum   = jnp.cumsum(residual_sq_series)                         # [r1, r1+r2, ...]
    past   = jnp.concatenate([jnp.array([0.0], DTYPE), csum[:-1]])  # sum_{k<i} r_k
    denom  = jnp.arange(N, dtype=DTYPE)
    denom  = jnp.maximum(denom, 1.0)                                # i=1 보호
    pm     = past / denom                                           # mean_{k<i}
    w      = jnp.exp(-jnp.asarray(eps, DTYPE) * pm)
    w      = jnp.clip(w, CAUSAL_CLIP_MIN, 1e6)
    return lax.stop_gradient(w)  # 논문 권고: w는 stop-gradient

def _ensure_pairwise_1d(t, z):
    """
    t, z를 '같은 길이의 1D 페어 벡터'로 변환.
    - 길이가 다르면 카테시안 곱(meshgrid) 후 ravel.
    - 길이가 같으면 그냥 평탄화해서 사용.
    """
    t = jnp.atleast_1d(t)
    z = jnp.atleast_1d(z)
    if t.size == z.size:
        return t.reshape(-1), z.reshape(-1)
    # 서로 길이가 다르면 카테시안 곱으로 모든 조합 평가
    T, Z = jnp.meshgrid(t.reshape(-1), z.reshape(-1), indexing='ij')
    return T.reshape(-1), Z.reshape(-1)

In [22]:
from flax import struct
import jax.numpy as jnp
from flax.training import train_state as flax_train_state


DTYPE = jnp.float64

@struct.dataclass
class SAWConfig:
    update_every: int = 1000   # 몇 step마다 λ 업데이트할지
    ema_beta: float = 0.9
    eps: float = 1e-12
    include_data: bool = True  # data term도 GradNorm에 포함할지
    normalize: bool = True     # λ 평균이 1이 되도록 정규화할지

def _default_saw():
    # r1,r2,r3,r4,data 각각에 대해 λ와 grad-norm EMA를 저장
    return {
        'lambda_r1': jnp.array(1.0, DTYPE),
        'lambda_r2': jnp.array(1.0, DTYPE),
        'lambda_r3': jnp.array(1.0, DTYPE),
        'lambda_r4': jnp.array(1.0, DTYPE),
        'lambda_data': jnp.array(1.0, DTYPE),

        'g_r1_ema': jnp.array(1.0, DTYPE),
        'g_r2_ema': jnp.array(1.0, DTYPE),
        'g_r3_ema': jnp.array(1.0, DTYPE),
        'g_r4_ema': jnp.array(1.0, DTYPE),
        'g_data_ema': jnp.array(1.0, DTYPE),
    }

class TrainState(flax_train_state.TrainState):
    saw: dict = struct.field(pytree_node=True,  default_factory=_default_saw)
    saw_cfg: SAWConfig = struct.field(pytree_node=False, default_factory=SAWConfig)


def _l2_norm_pytree(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.sqrt(sum([jnp.sum(jnp.square(x)) for x in leaves]))


def _grad_norm_of_term(params, batch, term: str):
    """
    term ∈ {'r1','r2','r3','r4','data'}
    """
    def term_loss(p):
        # compute_losses_unweighted는 아래에서 새로 정의할 버전
        Lp, Lb, Ld, extras = compute_losses_unweighted(p, batch)
        if term == 'r1':
            return extras['phys_r1']
        elif term == 'r2':
            return extras['phys_r2']
        elif term == 'r3':
            return extras['phys_r3']
        elif term == 'r4':
            return extras['phys_r4']
        elif term == 'data':
            return Ld
        else:
            raise ValueError(f"Unknown term for grad-norm: {term}")

    g = jax.grad(term_loss)(params)
    return _l2_norm_pytree(g)

def _update_saw(state: TrainState, params, batch):
    cfg = state.saw_cfg
    beta = jnp.array(cfg.ema_beta, DTYPE)
    eps  = jnp.array(cfg.eps, DTYPE)
    s    = state.saw

    # 1) 현재 step에서 각 항목별 grad-norm
    g_r1  = _grad_norm_of_term(params, batch, 'r1')
    g_r2  = _grad_norm_of_term(params, batch, 'r2')
    g_r3  = _grad_norm_of_term(params, batch, 'r3')
    g_r4  = _grad_norm_of_term(params, batch, 'r4')
    g_dat = _grad_norm_of_term(params, batch, 'data') if cfg.include_data else jnp.array(0.0, DTYPE)

    # 2) EMA 업데이트
    g_r1_ema  = beta * s['g_r1_ema']  + (1.0 - beta) * g_r1
    g_r2_ema  = beta * s['g_r2_ema']  + (1.0 - beta) * g_r2
    g_r3_ema  = beta * s['g_r3_ema']  + (1.0 - beta) * g_r3
    g_r4_ema  = beta * s['g_r4_ema']  + (1.0 - beta) * g_r4
    g_dat_ema = beta * s['g_data_ema'] + (1.0 - beta) * g_dat if cfg.include_data else jnp.array(0.0, DTYPE)

    # 3) GradNorm 수식: sum_g / g_k
    sum_g = g_r1_ema + g_r2_ema + g_r3_ema + g_r4_ema
    if cfg.include_data:
        sum_g = sum_g + g_dat_ema
    sum_g = sum_g + eps

    lam_r1   = sum_g / (g_r1_ema + eps)
    lam_r2   = sum_g / (g_r2_ema + eps)
    lam_r3   = sum_g / (g_r3_ema + eps)
    lam_r4   = sum_g / (g_r4_ema + eps)
    lam_data = sum_g / (g_dat_ema + eps) if cfg.include_data else jnp.array(1.0, DTYPE)

    # 4) λ 정규화 (평균이 1이 되도록)
    if cfg.normalize:
        n_terms = 4 + (1 if cfg.include_data else 0)
        denom = lam_r1 + lam_r2 + lam_r3 + lam_r4
        if cfg.include_data:
            denom = denom + lam_data
        denom = denom / n_terms

        lam_r1   = lam_r1   / denom
        lam_r2   = lam_r2   / denom
        lam_r3   = lam_r3   / denom
        lam_r4   = lam_r4   / denom
        lam_data = lam_data / denom if cfg.include_data else lam_data

    new_saw = {
        'lambda_r1': lam_r1, 'lambda_r2': lam_r2,
        'lambda_r3': lam_r3, 'lambda_r4': lam_r4,
        'lambda_data': lam_data,

        'g_r1_ema': g_r1_ema, 'g_r2_ema': g_r2_ema,
        'g_r3_ema': g_r3_ema, 'g_r4_ema': g_r4_ema,
        'g_data_ema': g_dat_ema,
    }
    return state.replace(saw=new_saw)


In [23]:
def compute_losses_unweighted(params, batch, causal_state=None):
    """Steady-state + Hard BC 버전
    - PDE: r1~r4 각각 MSE 계산 후 합쳐서 L_phys_total 구성
    - BC: Hard constraint이므로 L_bc_total = 0
    - Data: 있으면 MSE, 없으면 0
    """
    pnn = params['nn'] if (isinstance(params, dict) and 'nn' in params) else params
    def _safe_mean(x): return jnp.array(0.0, DTYPE) if (x.size == 0) else jnp.mean(x)

    # batch 언패킹 (N_b=0이므로 boundary 관련은 비어있을 것)
    (z_r,
     z_b0, z_b1,
     z_lab, n_lab, Te_lab, E_lab, G_lab,
     n_b0_lab, Te_b0_lab, E_b0_lab, G_b0_lab,
     n_b1_lab, Te_b1_lab, E_b1_lab, G_b1_lab,
     z_bpde0, z_bpde1) = batch

    # -------- Physics (PDE) Loss --------
    r1, r2, r3, r4 = v_residuals(pnn, z_r)
    L_phys_r1 = jnp.mean(r1**2)
    L_phys_r2 = jnp.mean(r2**2)
    L_phys_r3 = jnp.mean(r3**2)
    L_phys_r4 = jnp.mean(r4**2)

    # 내부 고정 weight 없이 동등하게 합산
    L_phys_total = L_phys_r1 + L_phys_r2 + L_phys_r3 + L_phys_r4
    # (평균 쓰고 싶으면 0.25 곱해도 됨)

    # -------- Boundary Loss (Hard BC → 0) --------
    L_bc_total = jnp.array(0.0, DTYPE)

    # -------- Data Loss (필요시) --------
    if (z_lab.size == 0):
        L_data_total = jnp.array(0.0, DTYPE)
        L_data_n = L_data_Te = L_data_E = L_data_G = jnp.array(0.0, DTYPE)
    else:
        N_hat_l, Te_hat_l, E_hat_l, G_hat_l = vmap(lambda zz: model_apply(pnn, zz))(z_lab.reshape(-1))
        n_hat_l = jnp.exp(N_hat_l)
        L_data_n  = jnp.mean((n_hat_l - n_lab.reshape(-1))**2)
        L_data_Te = jnp.mean((Te_hat_l - Te_lab.reshape(-1))**2)
        L_data_E  = jnp.mean((E_hat_l - E_lab.reshape(-1))**2)
        L_data_G  = jnp.mean((G_hat_l - G_lab.reshape(-1))**2)
        L_data_total = L_data_n + L_data_Te + L_data_E + L_data_G

    # extras: r1~r4 개별 loss를 그대로 남겨서 GradNorm/로깅에 사용
    extras = {
        'total': L_phys_total + L_data_total,  # Hard BC라 BC 제외
        'phys_r1': L_phys_r1, 'phys_r2': L_phys_r2,
        'phys_r3': L_phys_r3, 'phys_r4': L_phys_r4,
        # 예전 코드 호환용: phys_w는 이제 의미 없으니 1.0으로 고정
        'phys_w1': 1.0, 'phys_w2': 1.0, 'phys_w3': 1.0, 'phys_w4': 1.0,
        'bc_total': L_bc_total,
        # Hard BC에서는 모두 0이지만 플롯 호환성을 위해 키는 유지
        'bc_flux_L': jnp.array(0.0, DTYPE),
        'bc_flux_R': jnp.array(0.0, DTYPE),
        'bc_dTe_L': jnp.array(0.0, DTYPE),
        'bc_dTe_R': jnp.array(0.0, DTYPE),
        'bc_pde_flux_L': jnp.array(0.0, DTYPE),
        'bc_pde_flux_R': jnp.array(0.0, DTYPE),
        'bc_pde_dTe_L': jnp.array(0.0, DTYPE),
        'bc_pde_dTe_R': jnp.array(0.0, DTYPE),
        'data_total': L_data_total,
        'data_n': L_data_n, 'data_Te': L_data_Te,
        'data_E': L_data_E, 'data_G': L_data_G,
    }

    return L_phys_total, L_bc_total, L_data_total, extras


In [24]:
def create_train_state(key, lr=LR, b1=0.9, b2=0.999, weight_decay=0.0,
                       precondition_frequency=2, clip_norm=1.0,
                       use_decay=True, decay_steps=2000, decay_rate=0.9, staircase=False):
    # ★ steady: 입력은 (z,) → (1,1)
    net_vars = model.init(key, jnp.ones((1,1), dtype=DTYPE))

    if use_decay:
        lr_or_schedule = optax.warmup_exponential_decay_schedule(
            init_value=jnp.array(0.0, dtype=DTYPE),
            peak_value=jnp.array(1e-3, dtype=DTYPE),
            warmup_steps=22_000,
            transition_steps=2_000,
            decay_rate=0.9,
            staircase=True
        )
    else:
        lr_or_schedule = jnp.array(lr, dtype=DTYPE)

    tx = optax.chain(
        optax.clip_by_global_norm(clip_norm),
        soap(
            learning_rate=lr_or_schedule,
            b1=b1, b2=b2,
            weight_decay=weight_decay,
            precondition_frequency=precondition_frequency,
        )
    )

    return TrainState.create(
        apply_fn=None,
        params={'nn': net_vars},
        tx=tx,
        saw=_default_saw(),
        saw_cfg=SAWConfig(),  # include_ic=False 기본값(steady)
    )


In [25]:
# Parameter 해시 유틸 (RWF 대응 버전)
import hashlib, numpy as np
import jax.numpy as jnp
from flax.traverse_util import flatten_dict

def _hash_leaf(h, key, v):
    """leaf v를 타입별로 안전하게 해시"""
    # JAX/NumPy 배열
    if isinstance(v, (jnp.ndarray, np.ndarray)):
        arr = np.asarray(v)
        # dtype/shape까지 포함시켜 충돌 감소
        h.update(str(arr.dtype).encode())
        h.update(np.asarray(arr.shape, dtype=np.int64).tobytes())
        h.update(arr.tobytes())
        return

    # 튜플/리스트: 각 원소를 재귀적으로 해시
    if isinstance(v, (tuple, list)):
        h.update(f"{key}#len={len(v)}".encode())
        for i, item in enumerate(v):
            _hash_leaf(h, f"{key}/{i}", item)
        return

    # dict: 키 정렬 후 각 값 재귀 해시
    if isinstance(v, dict):
        for kk in sorted(v.keys()):
            _hash_leaf(h, f"{key}/{kk}", v[kk])
        return

    # None
    if v is None:
        h.update(b"None")
        return

    # 스칼라(파이썬 숫자 등)
    try:
        arr = np.asarray(v)
        h.update(str(arr.dtype).encode())
        h.update(np.asarray(arr.shape, dtype=np.int64).tobytes())
        h.update(arr.tobytes())
    except Exception:
        # 기타: 문자열 등
        h.update(str(v).encode())

def hash_params(tree):
    flat = flatten_dict(tree, sep='/')
    h = hashlib.sha256()
    for k, v in sorted(flat.items()):
        h.update(k.encode())
        _hash_leaf(h, k, v)  # [FIX] 타입 안전 해시
    return h.hexdigest()

from flax.training import checkpoints
from pathlib import Path
def save_params_npz(params, path):
    flat = flatten_dict(params, sep='/')
    out = {}
    for k, v in flat.items():
        if isinstance(v, (jnp.ndarray, np.ndarray)):
            out[k] = np.asarray(v)
        elif isinstance(v, (tuple, list)):
            for i, item in enumerate(v):
                out[f"{k}__{i}"] = np.asarray(item)
        else:
            # 스칼라/기타 가능성: 배열로 캐스팅 시도
            try:
                out[k] = np.asarray(v)
            except Exception:
                # 문자열 등은 저장 스킵하거나 문자열로 저장
                out[k] = np.asarray(str(v))
    np.savez(path, **out)

# [FIX] 튜플/리스트 leaf 지원 복원
def load_params_npz(path, like_params):
    data = np.load(path)
    flat_like = flatten_dict(like_params, sep='/')

    restored = {}
    # 1) 먼저 __i suffix가 있는 것들 묶기
    buckets = {}
    for key in data.files:
        if '__' in key and key.split('__')[-1].isdigit():
            base, idx = key.rsplit('__', 1)
            buckets.setdefault(base, {})[int(idx)] = data[key]
        else:
            restored[key] = data[key]

    # 2) 묶인 것들(튜플/리스트) 재구성 (인덱스 순서대로)
    for base, parts in buckets.items():
        restored[base] = tuple(parts[i] for i in sorted(parts.keys()))

    # 3) like_params의 구조를 따라 unflatten
    out = {}
    for k, _ in flat_like.items():
        if k in restored:
            v = restored[k]
        else:
            # 없으면 예외를 던지거나 기본값 사용
            raise KeyError(f"Missing key in npz: {k}")
        # unflatten
        node = out
        parts = k.split('/')
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = v
    return out

# 최초 state 만들기 (재실행 시 중복 초기화 방지)
# 최초 state 만들기
# >>> 새 TrainState 정의 후, 반드시 한 번 초기화 강제 <<<
if 'state' in globals():
    del state  # 오래된 타입의 state 제거

state = create_train_state(key, use_decay=True, decay_steps=2000)

try:
    state
    print("[skip] state already exists — not re-initializing")
except NameError:
    state = create_train_state(key, use_decay=True, decay_steps=2000)
    print("[init] state created")

# ckpt 복원 (필드 불일치 대비)
try:
    state = checkpoints.restore_checkpoint(str(CKPT_DIR), target=state)
except Exception as e:
    print("[warn] checkpoint restore failed (field mismatch). Starting fresh. err:", e)
print("[resume] step =", int(getattr(state, "step", 0)))
print("[resume] hash =", hash_params(state.params))



[skip] state already exists — not re-initializing
[resume] step = 0
[resume] hash = f13f706a662d6ef21bf44d4896823a41505e7a507471cc4406b473048b77cad9


In [26]:
@jax.jit
def train_step(state: TrainState, batch):
    def total_loss_fn(params):
        Lp, Lb, Ld, extras = compute_losses_unweighted(params, batch)

        # 개별 residual loss
        L_r1 = extras['phys_r1']
        L_r2 = extras['phys_r2']
        L_r3 = extras['phys_r3']
        L_r4 = extras['phys_r4']

        # SAW λ들
        lam = state.saw
        lam_r1 = lam['lambda_r1']
        lam_r2 = lam['lambda_r2']
        lam_r3 = lam['lambda_r3']
        lam_r4 = lam['lambda_r4']
        lam_d  = lam['lambda_data'] if state.saw_cfg.include_data else jnp.array(1.0, DTYPE)

        # GradNorm 기반 가중합
        total = (
            lam_r1 * L_r1 +
            lam_r2 * L_r2 +
            lam_r3 * L_r3 +
            lam_r4 * L_r4 +
            lam_d  * Ld
        )

        metrics = {
            'phys_total': Lp,      # r1~r4 합
            'bc_total': Lb,        # Hard BC → 0
            'data_total': Ld,
            **extras,
            'lambda_r1': lam_r1, 'lambda_r2': lam_r2,
            'lambda_r3': lam_r3, 'lambda_r4': lam_r4,
            'lambda_data': lam_d,
        }
        return total, metrics

    (loss, metrics), grads = jax.value_and_grad(total_loss_fn, has_aux=True)(state.params)
    new_state = state.apply_gradients(grads=grads)

    # 일정 step마다 SAW(GradNorm) 업데이트
    step_mod  = jnp.mod(new_state.step, new_state.saw_cfg.update_every)
    do_update = jnp.equal(step_mod, 0)

    def _upd(s):  return _update_saw(s, s.params, batch)
    def _keep(s): return s
    new_state = jax.lax.cond(do_update, _upd, _keep, new_state)

    return new_state, {'total': loss, **metrics}

In [27]:
log = []
best_loss = np.inf
start_step = int(getattr(state, "step", 0))

# (NEW) collocation 캐시
cached_z_r = None

for step in trange(start_step, EPOCHS, desc="training"):
    # (NEW) collocation resampling 여부 결정
    resample_r = (cached_z_r is None) or ((step % RESAMPLE_COLL_EVERY) == 0)
    z_r_fixed_arg = None if resample_r else cached_z_r

    # batch = sample_training_points_steady(
    #     jax.random.fold_in(key, step),
    #     N_r=N_r, N_b=N_b, N_lab=N_lab,
    #     z_minmax=z_minmax,
    #     z_data=z_ss, n_data=n_ss, Te_data=Te_ss, E_data=E_ss, G_data=G_ss,
    #     use_chebyshev=False, random_mode="uniform",
    #     use_boundary_data=True,
    #     boundary_tol=1e-8,          # 경계 매칭 완화(필요시)
    #     # N_bpde=max(1, N_b),         # ★ PDE 경계 샘플 확보 (NaN 방지)
    #     # (NEW) collocation 고정 전달
    #     z_r_fixed=z_r_fixed_arg,
    # )
    batch = sample_training_points_steady(
        jax.random.fold_in(key, step),
        N_r=N_r,                    # collocation points 유지
        N_b=0,                      # boundary points 불필요
        N_lab=N_lab,
        z_minmax=z_minmax,
        z_data=z_ss, n_data=n_ss, Te_data=Te_ss, E_data=E_ss, G_data=G_ss,
        use_chebyshev=False,
        random_mode="uniform",
        use_boundary_data=False,    # boundary 데이터 사용 안함
        boundary_tol=1e-8,
        N_bpde=0,                   # PDE boundary도 불필요
        z_r_fixed=z_r_fixed_arg,
    )

    # (NEW) 이번 스텝에 resample했다면 캐시 갱신
    if resample_r:
        cached_z_r = batch[0]  # 반환 튜플의 첫 항목이 z_r

    state, m = train_step(state, batch)

    if step % 200 == 0:
        phash = hash_params(state.params)
        record = {'step': step, 'params_hash': phash, **{k: float(v) for k, v in m.items()}}
        log.append(record)

        print(
            f"[{step:7d}] hash={phash[:10]} "
            f"total={record['total']:.3e} | "
            f"phys={record['phys_total']:.3e} "
            f"(r1={record['phys_r1']:.2e}, r2={record['phys_r2']:.2e}, r3={record['phys_r3']:.2e}, r4={record['phys_r4']:.2e}) | "
            f"bc={record['bc_total']:.3e} |  data={record['data_total']:.3e} | "
        )

        cur_loss = record['total']
        if (cur_loss < best_loss) and (step > 0):
            best_loss = cur_loss
            # checkpoints.save_checkpoint(str(CKPT_DIR), target=state, step=step, overwrite=True, keep=3)
            # save_params_npz(_vars_for_apply(state.params), BEST_PATH)
            # print(f"  ↳ [best] saved at step {step} | loss={best_loss:.3e}")
# (NEW) 저장: Orbax 시도 → 실패 시 Flax 직렬화 fallback
            try:
                checkpoints.save_checkpoint(str(CKPT_DIR), target=state,
                                             step=step, overwrite=True, keep=3)
                print(f"[ckpt] orbax saved (best) at step {step}")
            except Exception as e:
                print(f"[ckpt][warn] orbax failed (best): {e}")
                # Flax 직렬화 우회 저장
                out = CKPT_DIR / f"state_best_step{int(step)}.flax"
                with open(out, "wb") as f:
                    f.write(serialization.to_bytes(state))
                print(f"[ckpt][fallback] saved FLAX → {out}")
            # 파라미터 npz는 그대로
            save_params_npz(_vars_for_apply(state.params), BEST_PATH)
            print(f"  ↳ [best] saved at step {step} | loss={best_loss:.3e}")
    # =========================
    # [NEW] 주기적/지정 시점 저장 로직
    # =========================

    # (요청) 5만 epoch마다 결과 npz 저장 (참고: npz만 저장. ckpt는 저장하지 않음)
    if (step > 0) and (step % NPZ_SAVE_EVERY == 0):
        try:
            save_results_npz(state, log, batch, step, epoch_dir(step), data_path)
        except Exception as e:
            print(f"[npz][warn] failed at {step}: {e}")

    # (요청) 지정 epoch마다: 플롯(저장만) + npz 저장
    if step in PLOT_EPOCHS:
        try:
            # 플롯 저장 (표시 없음)
            save_all_plots_silent(state, log, step, epoch_dir(step), data_path)
            # npz도 함께 저장
            save_results_npz(state, log, batch, step, epoch_dir(step), data_path)
        except Exception as e:
            print(f"[plot/npz][warn] failed at {step}: {e}")

    # (요청) 10만 epoch마다: 별도 ckpt를 CKPT_SAVED_DIR에 저장 (기존 ckpt와 별개)
    if (step > 0) and (step % EXTRA_CKPT_EVERY == 0):
        try:
            from flax.training import checkpoints
            checkpoints.save_checkpoint(
                str(epoch_dir(step)), target=state, step=step, overwrite=True, keep=None
            )
            print(f"[ckpt-extra] saved(orbax) at {step} → {CKPT_SAVED_DIR}")
        except Exception as e:
            # fallback: FLAX bytes
            out = CKPT_SAVED_DIR / f"state_extra_{_tag(step)}.flax"
            with open(out, "wb") as f:
                f.write(serialization.to_bytes(state))
            print(f"[ckpt-extra][fallback] saved FLAX → {out} (err={e})")

            
    if (step % 2000 == 0) and (step > max(start_step, 0)) and (step > 0):
        try:
            checkpoints.save_checkpoint(str(CKPT_DIR), target=state,
                                         step=step, overwrite=True, keep=5)
            print(f"[ckpt] orbax saved (periodic) at step {step}")
        except Exception as e:
            print(f"[ckpt][warn] orbax failed (periodic): {e}")
            out = CKPT_DIR / f"state_periodic_step{int(step)}.flax"
            with open(out, "wb") as f:
                f.write(serialization.to_bytes(state))
            print(f"[ckpt][fallback] saved FLAX → {out}")


training:   0%|          | 1/300001 [00:33<2783:47:40, 33.41s/it]

[      0] hash=f13f706a66 total=6.847e+02 | phys=6.847e+02 (r1=3.50e-03, r2=2.65e+02, r3=9.74e-01, r4=4.19e+02) | bc=0.000e+00 |  data=0.000e+00 | 


training:   0%|          | 201/300001 [02:18<53:21:39,  1.56it/s]

[    200] hash=7f0b310204 total=6.025e+02 | phys=6.025e+02 (r1=2.71e-03, r2=2.53e+02, r3=1.01e+00, r4=3.49e+02) | bc=0.000e+00 |  data=0.000e+00 | 
[ckpt] orbax saved (best) at step 200
  ↳ [best] saved at step 200 | loss=6.025e+02


training:   0%|          | 401/300001 [04:09<58:42:14,  1.42it/s]

...

[ 124800] hash=4ac614919e total=3.838e-07 | phys=3.838e-07 (r1=6.49e-12, r2=3.84e-07, r3=4.98e-11, r4=3.41e-11) | bc=0.000e+00 |  data=0.000e+00 | 


training:  42%|████▏     | 125001/300001 [18:39:52<38:53:08,  1.25it/s]

[ 125000] hash=6c3e713d0d total=2.371e-07 | phys=2.371e-07 (r1=5.95e-12, r2=2.37e-07, r3=1.93e-11, r4=1.92e-11) | bc=0.000e+00 |  data=0.000e+00 | 
[npz] saved → /home/dlee/code/Plasma/Argon_steady_New_data_2510/Steady_BC_Hard_12_colloc0_make_gif_gradnorm_updated/CKPT_SAVED/step_0125000/results_step0125000.npz


training:  42%|████▏     | 125198/300001 [18:41:40<26:06:06,  1.86it/s]


KeyboardInterrupt: 

In [ ]:
#기존 log에 누락된 키들을 0으로 채워넣기
missing_keys = [
    'bc_flux_L', 'bc_flux_R', 'bc_dTe_L', 'bc_dTe_R',
    'bc_pde_flux_L', 'bc_pde_flux_R', 'bc_pde_dTe_L', 'bc_pde_dTe_R',
]

# log의 각 레코드에 누락된 키 추가
for record in log:
    for key in missing_keys:
        if key not in record:
            record[key] = 0.0
    
    # data 관련 키도 없으면 추가
    if 'data_n' not in record:
        record['data_n'] = 0.0
    if 'data_Te' not in record:
        record['data_Te'] = 0.0
    if 'data_E' not in record:
        record['data_E'] = 0.0
    if 'data_G' not in record:
        record['data_G'] = 0.0

In [ ]:
for record in log:
    if 'lambda_phys' not in record:
        record['lambda_phys'] = 1.0
    if 'lambda_bc' not in record:
        record['lambda_bc'] = 0.0
    if 'lambda_data' not in record:
        record['lambda_data'] = 0.0

In [ ]:
# =========================
# Common imports & helpers
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

# Te 양수 보장 (모델 출력 Te_raw → 물리 Te)
Te_FLOOR = 1e-8
def Te_pos(te_raw):
    return jax.nn.softplus(te_raw) + Te_FLOOR


# =========================
# 1) log(list of tuples) → DataFrame
#    (우리가 바꾼 트레이닝 루프의 로그 형식에 맞춤)
#    - 필수 열: step, total, phys_L, phys_sum_unweighted,
#              phys_rn2, phys_rG2, phys_re2, phys_rE2,
#              bc_n_sum, bc_T_sum, data_sum, ic_sum
#    - ALB=uncertainty 이면 뒤에 sigma_* 5개가 추가됨
# =========================



# =========================
# 2) 공통 플로팅 유틸
# =========================
def plot_many(df, ycols, title, ylog=True):
    plt.figure(figsize=(7, 4))
    for c in ycols:
        if c not in df.columns:  # 없는 열은 건너뛴다
            continue
        plt.plot(df['step'], df[c], label=c)
    plt.xlabel('step'); plt.ylabel('value')
    if ylog:
        plt.yscale('log')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# =========================
# 3) 로그 플롯 (요약 지표 위주)
# =========================
df_log = build_log_df(log)

# =========================
# Steady-state: loss plot bundle
# =========================
def _all_zero_or_missing(df, cols):
    cols = [c for c in cols if c in df.columns]
    if not cols: 
        return True
    return float(np.nan_to_num(df[cols].to_numpy())).sum() == 0.0

def plot_all_losses_steady(df):
    # 1) Total & grouped
    cols = ['total', 'phys_total', 'bc_total', 'data_total']
    # IC는 steady에선 보통 0 → 전부 0이 아니면 표시
    if not _all_zero_or_missing(df, ['ic_total']):
        cols.append('ic_total')
    plot_many(df, cols, title='[Plot 1] Total & Grouped Sums (steady)', ylog=True)

    # 2) Physics breakdown
    plot_many(
        df, ['phys_total','phys_r1','phys_r2','phys_r3','phys_r4'],
        title='[Plot 2] Physics Residual Breakdown (steady)', ylog=True
    )

    # 3) Physics weights (내부 리포팅용)
    plot_many(
        df, ['phys_w1','phys_w2','phys_w3','phys_w4'],
        title='[Plot 3] Physics Residual weight (steady)', ylog=False
    )

    # 4) SAW lambdas (IC 제외가 기본)
    cols = ['lambda_phys', 'lambda_bc', 'lambda_data']
    if 'lambda_ic' in df.columns and not _all_zero_or_missing(df, ['lambda_ic']):
        cols.append('lambda_ic')
    plot_many(df, cols, title='[Plot 4] SAW λ (steady)', ylog=True)

    # 5) Data breakdown (있을 때만 유의미)
    plot_many(
        df, ['data_total', 'data_n', 'data_Te', 'data_E', 'data_G'],
        title='[Plot 5] Data Loss Breakdown (steady)', ylog=True
    )

    # 6) IC breakdown: steady에선 대개 전부 0 → 전부 0이 아니면 표시
    if not _all_zero_or_missing(df, ['ic_total','ic_N','ic_Te']):
        plot_many(
            df, ['ic_total', 'ic_N', 'ic_Te'],
            title='[Plot 6] Initial Condition Loss Breakdown (steady)', ylog=True
        )

df_log = build_log_df(log)
plot_all_losses_steady(df_log)


# data_path 재지정 
(z_ss, n_ss, Te_ss, E_ss, G_ss), z_minmax, zfinal, t_minmax, Tfinal = load_csv_steady(data_path)
Tfinal = Tfinal
Lz     = zfinal
# =========================
# 4) PINN 2D 맵 (n_i, T_e) / C-시뮬레이터 단면 비교
#    - 모델의 출력: (N_hat, Te_hat_raw, E_hat, G_hat)
#    - 물리변환: n = exp(N_hat), Te = softplus(Te_raw)+eps
# =========================
# =========================
# Steady-state: Exact vs PINN (single overlay per field)
# =========================
def _tail_average_by_z(df, tail_k=None):
    """마지막 tail_k개 시점 평균(없으면 최종시각 단일 스냅샷). z-정렬 반환."""
    if (tail_k is None) or (tail_k <= 1):
        t_star = df['time[s]'].max()
        sub = df.loc[df['time[s]'] == t_star].copy()
        return t_star, sub.sort_values('z[m]')
    # tail 평균
    ut = np.sort(df['time[s]'].unique())
    sel = ut[-int(tail_k):]
    sub = df[df['time[s]'].isin(sel)].copy()
    g = sub.groupby('z[m]', as_index=False)[['ni_norm','Te_norm','E_norm','Fi_norm']].mean()
    return float(sel.mean()), g.sort_values('z[m]')

def _overlay_profile(z_SI, exact, pred, title, ylab):
    plt.figure(figsize=(7,4))
    plt.plot(z_SI, exact, marker='o', ms=3, lw=0, label='Exact (C-sim)')
    plt.plot(z_SI, pred,  lw=1.8, label='PINN')
    plt.xlabel('z [m]'); plt.ylabel(ylab); plt.title(title)
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

def compare_profiles_steady(state, data_path, tail_k=None):
    """
    tail_k: 마지막 K개 시점 평균(없으면 최종시각 한 장)
    필요한 상수: l0, t0
    model_apply(params, z̄)  # z-only
    """
    df = pd.read_csv(data_path)
    t_star_SI, sub = _tail_average_by_z(df, tail_k=tail_k)

    # Exact (CSV)
    z_SI   = sub['z[m]'].to_numpy()
    ni_sim = sub['ni_norm'].to_numpy()
    Te_sim = sub['Te_norm'].to_numpy()
    E_sim  = sub['E_norm'].to_numpy()
    G_sim  = sub['Fi_norm'].to_numpy()

    # PINN (z-only)
    z_bar = jnp.asarray(z_SI / l0, dtype=DTYPE)
    Np, Tep, Ep, Gp = jax.vmap(lambda zz: model_apply(state.params['nn'], zz))(z_bar)
    ni_pinn = np.asarray(jnp.exp(Np))
    Te_pinn = np.asarray(Tep)
    E_pinn  = np.asarray(Ep)
    G_pinn  = np.asarray(Gp)

    tag = f" @ steady (t ≈ {t_star_SI:.2e} s)" if tail_k is None or tail_k<=1 else f" @ tail-avg K={tail_k} (t̄ ≈ {t_star_SI:.2e} s)"
    _overlay_profile(z_SI, ni_sim, ni_pinn, "Ion Density $n_i/n_0$"+tag, r"$n_i/n_0$")
    _overlay_profile(z_SI, Te_sim, Te_pinn, "Electron Temperature $T_e/T_0$"+tag, r"$T_e/T_0$")
    _overlay_profile(z_SI, E_sim,  E_pinn,  "Electric Field $E$"+tag, r"$E$")
    _overlay_profile(z_SI, G_sim,  G_pinn,  r"Ion Flux $\Gamma$"+tag, r"$\Gamma$")
# 한 장 스냅샷(최종시각)
compare_profiles_steady(state, data_path)

# 마지막 5개 시점 평균(더 steady스럽게)
# compare_profiles_steady(state, data_path, tail_k=5)

# 로그 플롯(steady)
df_log = build_log_df(log)
plot_all_losses_steady(df_log)

# Exact vs PINN (steady overlay)
compare_profiles_steady(state, data_path)
# compare_profiles_steady(state, data_path, tail_k=5)



# DONE